# 02 — Expérimentations Preprocessing

Notebook de test avant d'écrire le code propre dans `src/preprocess.py`.  
L'idée c'est de tester chaque étape ici, voir si ça marche, puis la déplacer.

Ce qu'on va tester :
1. Chargement et fusion des 4 fichiers CSV
2. Nettoyage du texte
3. Combinaison titre + corps
4. Train / Val / Test split
5. Tokenisation pour le BiLSTM (vocabulaire manuel)
6. Tokenisation pour RoBERTa (HuggingFace)
7. Vérification que tout est bon avant de passer à preprocess.py

## 0 — Imports

In [1]:
import os
import sys
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

from transformers import RobertaTokenizer

# on remonte d'un niveau pour accéder à config.py
sys.path.append(os.path.join(os.getcwd(), '..'))
import config

print('Imports OK')

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


---
## 1 — Chargement des données

On recharge les 4 fichiers comme dans le notebook d'exploration.  
Cette logique sera copiée dans `preprocess.py` une fois validée.

In [2]:
frames = []

for key, filename in config.RAW_FILES.items():
    path = os.path.join(config.DATA_RAW_DIR, filename)
    df_temp = pd.read_csv(path)

    # on ajoute le label et la source depuis config
    df_temp['label']  = config.FILE_LABELS[key]
    df_temp['source'] = config.FILE_SOURCES[key]

    frames.append(df_temp)

df = pd.concat(frames, ignore_index=True)

# on garde seulement les colonnes utiles
df = df[['title', 'text', 'label', 'source']]

print(f'Dataset chargé : {df.shape}')
df.head(3)

Dataset chargé : (422, 4)


,title,text,label,source
0,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,0,buzzfeed
1,Charity: Clinton Foundation Distributed “Water...,Former President Bill Clinton and his Clinton ...,0,buzzfeed
2,A Hillary Clinton Administration May be Entire...,After collapsing just before trying to step in...,0,buzzfeed


---
## 2 — Nettoyage du texte

On teste différentes opérations de nettoyage et on regarde l'effet sur quelques exemples.  
On ne veut pas supprimer trop d'info — juste le bruit.

In [3]:
# fonction de nettoyage de base
def clean_text(text):
    if pd.isna(text):
        return ''

    # minuscules
    text = text.lower()

    # suppression des URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # suppression des emails
    text = re.sub(r'\S+@\S+', '', text)

    # suppression des caractères spéciaux (on garde les lettres et chiffres)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# test sur quelques exemples
test_samples = [
    "Check out this AMAZING story! http://fakenews.com/article?id=123 #politics",
    "BREAKING: President signs bill. Contact us at info@news.com for more details!!!",
    df['text'].iloc[0]
]

for s in test_samples:
    print('AVANT :', s[:120])
    print('APRÈS :', clean_text(s)[:120])
    print()

AVANT : Check out this AMAZING story! http://fakenews.com/article?id=123 #politics
APRÈS : check out this amazing story politics

AVANT : BREAKING: President signs bill. Contact us at info@news.com for more details!!!
APRÈS : breaking president signs bill contact us at for more details

AVANT : I woke up this morning to find a variation of this headline splashed all over my news feed:

Bill Clinton: ‘Natural’ For
APRÈS : i woke up this morning to find a variation of this headline splashed all over my news feed bill clinton natural for foun



In [4]:
# on applique le nettoyage aux deux colonnes
df['title_clean'] = df['title'].apply(clean_text)
df['text_clean']  = df['text'].apply(clean_text)

print('Nettoyage appliqué')
df[['title', 'title_clean', 'text_clean']].head(3)

Nettoyage appliqué


,title,title_clean,text_clean
0,Proof The Mainstream Media Is Manipulating The...,proof the mainstream media is manipulating the...,i woke up this morning to find a variation of ...
1,Charity: Clinton Foundation Distributed “Water...,charity clinton foundation distributed watered...,former president bill clinton and his clinton ...
2,A Hillary Clinton Administration May be Entire...,a hillary clinton administration may be entire...,after collapsing just before trying to step in...


---
## 3 — Combinaison titre + corps

On a décidé dans l'exploration d'utiliser les deux.  
On teste le template défini dans `config.py` : `"{title} [SEP] {text}"`

In [5]:
# combinaison avec le template de config
df['combined'] = df.apply(
    lambda row: config.TEXT_COMBINATION_TEMPLATE.format(
        title=row['title_clean'],
        text=row['text_clean']
    ),
    axis=1
)

# vérification
print('Exemple de texte combiné :')
print(df['combined'].iloc[0][:300])
print()

# longueur en mots après combinaison
df['combined_len'] = df['combined'].apply(lambda x: len(x.split()))
print('Longueur moyenne (mots) :', df['combined_len'].mean().round(1))
print('Longueur max   (mots) :', df['combined_len'].max())
print('Longueur médiane (mots):', df['combined_len'].median())

Exemple de texte combiné :
proof the mainstream media is manipulating the election by taking bill clinton out of context [SEP] i woke up this morning to find a variation of this headline splashed all over my news feed bill clinton natural for foundation donors to seek favors here s google naturally my reaction was oh s t what

Longueur moyenne (mots) : 623.2
Longueur max   (mots) : 5671
Longueur médiane (mots): 420.5


---
## 4 — Train / Val / Test Split

On vérifie que la répartition des labels est bien maintenue dans chaque split (stratify).  
Les ratios viennent de `config.py` : 80 / 10 / 10.

In [6]:
# premier split : train vs (val + test)
X = df['combined'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=(1 - config.TRAIN_SIZE),
    random_state=config.RANDOM_SEED,
    stratify=y
)

# deuxième split : val vs test (50/50 du reste)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=config.RANDOM_SEED,
    stratify=y_temp
)

print(f'Train : {len(X_train)} samples  | labels: {Counter(y_train)}')
print(f'Val   : {len(X_val)}  samples  | labels: {Counter(y_val)}')
print(f'Test  : {len(X_test)} samples  | labels: {Counter(y_test)}')

Train : 337 samples  | labels: Counter({np.int64(1): 169, np.int64(0): 168})
Val   : 42  samples  | labels: Counter({np.int64(1): 21, np.int64(0): 21})
Test  : 43 samples  | labels: Counter({np.int64(0): 22, np.int64(1): 21})


---
## 5 — Vocabulaire pour le BiLSTM

Le BiLSTM a besoin d'un vocabulaire construit à la main.  
On ne construit le vocabulaire que sur le **train set** — jamais sur val ou test.  
Deux tokens spéciaux : `<PAD>` pour le padding, `<UNK>` pour les mots inconnus.

In [7]:
# construction du vocabulaire depuis le train uniquement
word_counts = Counter()

for text in X_train:
    word_counts.update(text.split())

print(f'Mots uniques dans le train : {len(word_counts)}')
print(f'Top 10 mots : {word_counts.most_common(10)}')

Mots uniques dans le train : 13023
Top 10 mots : [('the', 10471), ('to', 5737), ('and', 4939), ('a', 4690), ('of', 4639), ('in', 3932), ('that', 3038), ('s', 2934), ('is', 2089), ('trump', 1966)]


In [8]:
# on garde seulement les mots qui apparaissent au moins MIN_WORD_FREQ fois
vocab = ['<PAD>', '<UNK>']  # tokens spéciaux en premier

for word, count in word_counts.items():
    if count >= config.MIN_WORD_FREQ:
        vocab.append(word)

# on tronque si le vocab dépasse la taille max
vocab = vocab[:config.MAX_VOCAB_SIZE]

# dictionnaire mot → index
word2idx = {word: idx for idx, word in enumerate(vocab)}

print(f'Taille du vocabulaire final : {len(word2idx)}')
print(f'Index de <PAD> : {word2idx["<PAD>"]}')
print(f'Index de <UNK> : {word2idx["<UNK>"]}')
print(f'Exemple - "president" : {word2idx.get("president", "not in vocab")}')

Taille du vocabulaire final : 8827
Index de <PAD> : 0
Index de <UNK> : 1
Exemple - "president" : 59


In [9]:
# fonction pour convertir un texte en séquence d'indices
def text_to_indices(text, word2idx, max_len):
    tokens = text.split()[:max_len]  # on tronque si trop long
    indices = [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

    # padding si trop court
    if len(indices) < max_len:
        indices += [word2idx['<PAD>']] * (max_len - len(indices))

    return indices


# test sur un exemple
sample_text  = X_train[0]
sample_indices = text_to_indices(sample_text, word2idx, config.MAX_SEQUENCE_LENGTH)

print(f'Texte (50 premiers mots) : {" ".join(sample_text.split()[:10])} ...')
print(f'Indices (10 premiers)    : {sample_indices[:10]}')
print(f'Longueur de la séquence  : {len(sample_indices)}')

Texte (50 premiers mots) : declassified docs show that obama admin created isis in 2012 ...
Indices (10 premiers)    : [1, 1, 2, 3, 4, 1, 5, 6, 7, 8]
Longueur de la séquence  : 256


---
## 6 — Tokenisation RoBERTa

RoBERTa a son propre tokenizer — on n'utilise pas notre vocabulaire ici.  
On vérifie juste que le tokenizer fonctionne et que les sorties ont la bonne forme.

In [ ]:
# chargement du tokenizer (télécharge ~500MB la première fois)
tokenizer = RobertaTokenizer.from_pretrained(config.ROBERTA_MODEL_NAME)

print(f'Tokenizer chargé : {config.ROBERTA_MODEL_NAME}')
print(f'Taille du vocabulaire RoBERTa : {tokenizer.vocab_size}')

In [ ]:
# test de tokenisation sur un exemple
sample = X_train[0]

encoded = tokenizer(
    sample,
    max_length=config.ROBERTA_MAX_LENGTH,
    padding='max_length',
    truncation=True,
    return_tensors='pt'  # retourne des tenseurs PyTorch
)

print('Clés retournées :', list(encoded.keys()))
print('input_ids shape :', encoded['input_ids'].shape)
print('attention_mask shape :', encoded['attention_mask'].shape)
print()

# vérification : les 1 dans attention_mask = vrais tokens, les 0 = padding
n_real_tokens = encoded['attention_mask'].sum().item()
print(f'Vrais tokens : {n_real_tokens} / {config.ROBERTA_MAX_LENGTH}')

In [ ]:
# test sur un batch pour simuler ce que le DataLoader fera
batch_texts = list(X_train[:4])

batch_encoded = tokenizer(
    batch_texts,
    max_length=config.ROBERTA_MAX_LENGTH,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print(f'Batch de {len(batch_texts)} textes')
print(f'input_ids shape : {batch_encoded["input_ids"].shape}')  # (4, 256)
print(f'Tout est bon si shape = ({len(batch_texts)}, {config.ROBERTA_MAX_LENGTH})')

---
## 7 — Résumé et décisions

Tout ce qu'on a validé ici va passer dans `src/preprocess.py` :

| Étape | Statut | Notes |
|---|---|---|
| Chargement des 4 CSV | ✅ | labels depuis les noms de fichiers |
| Nettoyage texte | ✅ | lower, URLs, spéciaux, espaces |
| Combinaison titre + texte | ✅ | template `[SEP]` |
| Split 80/10/10 stratifié | ✅ | labels équilibrés dans chaque split |
| Vocabulaire BiLSTM | ✅ | construit sur train seulement |
| Tokenizer RoBERTa | ✅ | shape correcte, padding OK |

---
**Prochaine étape** : écrire `src/preprocess.py` avec tout ce qu'on a testé ici.